# Chargement & Nettoyage des Données
**Équipe 12** — Nino Tissot · Audrey Nourry · Mailis Briens · Hajar Belgroun

Ce notebook couvre le chargement du dataset brut, le filtrage sectoriel pharma/biotech, 
le nettoyage des colonnes financières et l'inspection des valeurs manquantes.

Fichier source : `investments_VC.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, ConfusionMatrixDisplay

import requests
import time
import re
from typing import List, Dict
from bs4 import BeautifulSoup

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving investments_VC.csv to investments_VC.csv


## 1. Chargement et EDA

In [ ]:
df = pd.read_csv("investments_VC.csv", encoding='latin1')
df = df.rename(columns={' market ': "market", ' funding_total_usd ': "funding_total_usd"})
print(f"Dataset complet : {len(df)} lignes, {df.shape[1]} colonnes")

Dataset complet : 54294 lignes, 39 colonnes


In [ ]:
pharma_keywords = ['pharma', 'biotech', 'abiotechnology', 'biopharmaceutical',
                  'drug', 'therapeutics', 'medicine', 'clinical',
                  'genomics', 'diagnostic', 'medtech', 'life science',
                  'oncology', 'immunology', 'neuroscience', 'vaccine']

mask = pd.Series([False] * len(df))
for col in ['category_list', 'market']:
    if col in df.columns:
        for kw in pharma_keywords:
            mask |= df[col].astype(str).str.lower().str.contains(kw, na=False)

pharma_df = df[mask].copy()
print(f"Startups pharma/biotech : {len(pharma_df)}")

Startups pharma/biotech : 4207


In [ ]:
pharma_df['funding_total_usd'] = (pharma_df['funding_total_usd']
    .astype(str).str.strip()
    .str.replace(',', '', regex=False)
    .replace('-', np.nan).replace('', np.nan))
pharma_df['funding_total_usd'] = pd.to_numeric(pharma_df['funding_total_usd'], errors='coerce')

In [ ]:
print("Répartition des statuts :")
print(pharma_df['status'].value_counts())
print(f"\nTaux de valeurs manquantes (top 10) :")
print(pharma_df.isnull().mean().sort_values(ascending=False).head(10).apply(lambda x: f"{x:.1%}"))

Répartition des statuts :
status
operating    3730
acquired      225
closed        147
Name: count, dtype: int64

Taux de valeurs manquantes (top 10) :
founded_year         32.1%
founded_quarter      32.1%
founded_month        32.1%
founded_at           32.0%
state_code           23.6%
homepage_url          9.0%
city                  5.0%
funding_total_usd     5.0%
region                3.9%
country_code          3.9%
dtype: object


In [ ]:
print(pharma_df['status'].unique())

['operating' 'acquired' 'closed' nan]
